# Orchestrator E2E Demo

这份 notebook 用来测试完整链路：`diagnosis_orchestrator -> diagnosis_agent -> confirmation -> coach_agent`。

In [1]:
import json

from diagnosis_orchestrator import FoundryDiagnosisOrchestrator, orchestrator_environment

print(orchestrator_environment())

{'orchestrator_version': '2026-06-17-diagnosis-orchestrator-v1', 'project_endpoint': 'https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default', 'model_deployment': 'gpt-4o-mini', 'structured_outputs_supported': 'True'}


In [2]:
PROBLEM_TEXT = "已知正实数 a,b 满足 a+2b=1，求 1/a + 1/b 的最小值。"

REFERENCE_ANSWER = (
    "由约束 a=1-2b，原式可化为 1/(1-2b)+1/b。"
    "更稳的做法是用柯西不等式或基本不等式："
    "(1/a+1/b)(a+2b) >= (1+sqrt(2))^2。"
    "因为 a+2b=1，所以 1/a+1/b >= (1+sqrt(2))^2。"
    "当 a:b=1:sqrt(2) 且 a+2b=1 时取等，最小值为 (1+sqrt(2))^2。"
)

STUDENT_PROFILE = "学生会基础代数变形，但遇到最值题时常不知道该先用代换、均值不等式还是柯西。"

INITIAL_STUDENT_ANSWER = "我先想把 a 换成 1-2b，但是后面不知道怎么求最小值，也不知道该用哪个不等式。"

CONFIRMATION_REPLY = "不是算错，也不是看错题，我是真不知道这题第一步之后该用什么方法求最小值。"

In [3]:
def dump_orchestrator_result(label, result):
    print(f"\n=== {label} ===")
    print("action:", result.action)
    print("content:", result.content)
    print("done:", result.done)
    print("stop_reason:", result.stop_reason)
    if result.diagnosis is not None:
        print("diagnosis:")
        print(json.dumps(result.diagnosis.as_dict(), ensure_ascii=False, indent=2))
    if result.confirmation_analysis is not None:
        print("confirmation_analysis:")
        print(json.dumps(result.confirmation_analysis.as_dict(), ensure_ascii=False, indent=2))
    if result.coach_initial_student_reply is not None:
        print("coach_initial_student_reply:", result.coach_initial_student_reply)
    if result.coach_handoff is not None:
        print("coach_handoff:\n" + result.coach_handoff)


In [4]:
orchestrator = FoundryDiagnosisOrchestrator()

flow_session = orchestrator.create_session(
    problem_text=PROBLEM_TEXT,
    reference_answer=REFERENCE_ANSWER,
    student_profile=STUDENT_PROFILE,
    coach_max_turns=4,
    max_confirm_turns=2,
    direct_to_coach_confidence=0.85,
)

first = orchestrator.start(INITIAL_STUDENT_ANSWER, session=flow_session)
dump_orchestrator_result("Orchestrator Turn 1", first)

final_result = first


=== Orchestrator Turn 1 ===
action: ask_confirmation
content: 我现在更怀疑你不是不会知识点，而是不知道第一步该抓哪个中间量。你的卡点更接近这个吗？如果不是，请直接说你真正卡在哪。
done: False
stop_reason: await_confirmation
diagnosis:
{
  "problem_text": "已知正实数 a,b 满足 a+2b=1，求 1/a + 1/b 的最小值。",
  "reference_answer": "由约束 a=1-2b，原式可化为 1/(1-2b)+1/b。更稳的做法是用柯西不等式或基本不等式：(1/a+1/b)(a+2b) >= (1+sqrt(2))^2。因为 a+2b=1，所以 1/a+1/b >= (1+sqrt(2))^2。当 a:b=1:sqrt(2) 且 a+2b=1 时取等，最小值为 (1+sqrt(2))^2。",
  "student_answer": "我先想把 a 换成 1-2b，但是后面不知道怎么求最小值，也不知道该用哪个不等式。",
  "student_profile": "学生会基础代数变形，但遇到最值题时常不知道该先用代换、均值不等式还是柯西。",
  "error_type": "missing_strategy",
  "confidence": 0.97,
  "reason": "学生已经会代换，但不知道接下来该使用哪种不等式求最小值，缺乏求最值的整体思路。",
  "evidence": "后面不知道怎么求最小值，也不知道该用哪个不等式。",
  "coach_mode": "socratic_light",
  "coach_trap": "学生知道局部知识，但不会先确定中间目标。",
  "coach_prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。",
  "source": "ai_tool_json_mode"
}


In [5]:
if first.action != "enter_coach":
    second = orchestrator.handle_student_reply(
        CONFIRMATION_REPLY,
        session=flow_session,
    )
    dump_orchestrator_result("Orchestrator Turn 2", second)
    final_result = second


=== Orchestrator Turn 2 ===
action: enter_coach
content: 已完成错因确认，进入 coach。
done: True
stop_reason: enter_coach_after_confirmation
diagnosis:
{
  "problem_text": "已知正实数 a,b 满足 a+2b=1，求 1/a + 1/b 的最小值。",
  "reference_answer": "由约束 a=1-2b，原式可化为 1/(1-2b)+1/b。更稳的做法是用柯西不等式或基本不等式：(1/a+1/b)(a+2b) >= (1+sqrt(2))^2。因为 a+2b=1，所以 1/a+1/b >= (1+sqrt(2))^2。当 a:b=1:sqrt(2) 且 a+2b=1 时取等，最小值为 (1+sqrt(2))^2。",
  "student_answer": "我先想把 a 换成 1-2b，但是后面不知道怎么求最小值，也不知道该用哪个不等式。",
  "student_profile": "学生会基础代数变形，但遇到最值题时常不知道该先用代换、均值不等式还是柯西。",
  "error_type": "missing_strategy",
  "confidence": 0.96,
  "reason": "学生已经会代换，但不知道接下来该使用哪种不等式求最小值，缺乏解题策略。",
  "evidence": "后面不知道怎么求最小值，也不知道该用哪个不等式。",
  "coach_mode": "socratic_light",
  "coach_trap": "学生知道局部知识，但不会先确定中间目标。",
  "coach_prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。",
  "source": "ai_tool_json_mode"
}
confirmation_analysis:
{
  "stance": "confirm",
  "has_new_reason": false,
  "extracted_reason": "",
  "confidence": 0.97,
  "reason": "学生明确表示不知道第一步之后该用什么方法求最小值

In [6]:
if final_result.action == "enter_coach" and final_result.coach_session is not None:
    coach = orchestrator.coach_agent
    coach_result = coach.start(session=final_result.coach_session)
    print("\n=== Coach Turn 1 ===")
    print("coach_input:", final_result.coach_initial_student_reply)
    print(coach_result.content)
    print("quality:", coach_result.reply_quality.value)
    print("completed:", coach_result.reply_analysis.completed)
    print("mode:", coach_result.strategy.mode.value)
    print("analysis_source:", coach_result.reply_analysis.source)
    print("analysis_reason:", coach_result.reply_analysis.reason)
    print("stop_reason:", coach_result.stop_reason)



=== Coach Turn 1 ===
coach_input: 不是算错，也不是看错题，我是真不知道这题第一步之后该用什么方法求最小值。
别着急，先把 a 用 b 表示出来吧——因为 a+2b=1 可以得到 a=1-2b。把这个代进去，看看 1/a+1/b 会变成只含 b 的式子，然后再想怎么让它最小。你觉得接下来可以怎么处理这个只含 b 的表达式？
quality: empty
completed: False
mode: socratic_light
analysis_source: ai_tool_text_json_validated
analysis_reason: 学生只表示不知道该用什么方法，未给出任何思路或步骤，未展示对题目关键求最小值的理解。
stop_reason: continue
